In [ ]:
# to print nvidia driver's status table + to confirm that a gpu is attached to this session
!nvidia-smi

In [ ]:
from google.colab import userdata, drive
import os

USERNAME = "mardyweb"
REPO     = "atml-pa0"
TOKEN    = userdata.get('GITHUB_TOKEN')

# storing the url in an environment variable:
os.environ['GIT_URL'] = f"https://{TOKEN}@github.com/{USERNAME}/{REPO}.git"

# clones the repo only if it is not already here (safe to re-run):
if not os.path.exists(f"/content/{REPO}"):
    !git clone $GIT_URL
%cd /content/$REPO

!git config user.email "maryamw17@outlook.com"
!git config user.name "Maryam"

In [ ]:
from utils import set_seed, get_device, subset_loaders, save_results, save_fig
import matplotlib.pyplot as plt

# should print cuda:
print(get_device())

In [ ]:
!pip install -q transformers

from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import torch, requests
from utils import set_seed, get_device, save_results, save_fig

set_seed(42)
device = get_device()

# ViT-Base/16: 86M params, 12 encoder layers, 12 attention heads per layer, 768-dim embeddings, 16x16 patches, pre-trained on ImageNet-21k and
# fine-tuned on ImageNet-1k (so it predicts the standard 1000 classes)
MODEL = "google/vit-base-patch16-224"

# The processor handles resizing to 224x224 and normalising with the exact mean/std the model was trained on:
processor = ViTImageProcessor.from_pretrained(MODEL)
model_vit = ViTForImageClassification.from_pretrained(MODEL).to(device)
model_vit.eval()

print(f"patch size: {model_vit.config.patch_size}")
print(f"hidden dim: {model_vit.config.hidden_size}")
print(f"layers: {model_vit.config.num_hidden_layers}, "
      f"heads: {model_vit.config.num_attention_heads}")
print(f"output classes: {model_vit.config.num_labels}")

In [ ]:
from google.colab import files
uploaded = files.upload()   # opens a file picker so i can select 3 images

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

images = {}
for fname in uploaded:
    # "mycat.jpg" -> "mycat" (removes file type)
    name = fname.rsplit('.', 1)[0]
    images[name] = Image.open(fname).convert("RGB")
    print(f"{name}: {images[name].size}")

fig, ax = plt.subplots(1, len(images), figsize=(4*len(images), 4))
if len(images) == 1: ax = [ax]
for a, (name, im) in zip(ax, images.items()):
    a.imshow(im); a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
results_cls = {}

for name, img in images.items():
    # return_tensors="pt" gives PyTorch tensors, the processor resizes to 224x224 and normalises
    inputs = processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_vit(**inputs)

    # logits: [1, 1000]: one raw score per ImageNet class.
    logits = outputs.logits
    probs  = logits.softmax(dim=-1)[0]

    top5 = probs.topk(5)
    top1_idx = top5.indices[0].item()
    # id2label maps class index to a human-readable name
    top1_label = model_vit.config.id2label[top1_idx]

    print(f"\n{name}  ->  {top1_label}  ({top5.values[0].item():.4f})")
    print("  top-5:")
    for p, i in zip(top5.values, top5.indices):
        print(f"    {model_vit.config.id2label[i.item()]:<35} {p.item():.4f}")

    results_cls[name] = {
        "top1_label": top1_label,
        "top1_prob": round(top5.values[0].item(), 4),
        "top5": [{"label": model_vit.config.id2label[i.item()],
                  "prob": round(p.item(), 4)}
                 for p, i in zip(top5.values, top5.indices)]
    }

save_results("task2_classification", results_cls)

In [ ]:
# committing + pushing to github
!git config --global core.editor true
!git add .
!git commit -m "Task 2.1: ViT classification"
!git pull --no-rebase --no-edit $GIT_URL main
!git push $GIT_URL

In [ ]:
from google.colab import _message
import json, os

nb = _message.blocking_request('get_ipynb', timeout_sec=60)['ipynb']
os.makedirs('notebooks', exist_ok=True)
with open('notebooks/task2_vit.ipynb', 'w') as f:
    json.dump(nb, f, indent=1)

!git add notebooks/task2_vit.ipynb
!git commit -m "Task 2.1: ViT notebook"
!git pull --no-rebase --no-edit $GIT_URL main
!git push $GIT_URL